# DEMONEXT CCD Camera Dark Calibration

Acquire a seuqence of biases and darks while stepping the CCD temperature from 0.0 to -20.0 C

## WARNING: READ THIS

**Never, Ever, Run All in this notebook at once**  

This was designed to test individula telescope operations and document them. Run without care you could
try to drive the telescope places you don't want it to go.

In [2]:
import sys
import os
import time
import math
import glob
import datetime
from datetime import date, timedelta

# Windows Component Object Model (COM) client module.

from win32com.client import Dispatch

# modules we need from anaconda

import numpy as np

# FITS writing and handling

from astropy.io import fits

# astropy units, coordinates, and times

from astropy import units as u
from astropy.coordinates import SkyCoord
from astropy.coordinates import TETE
from astropy.time import Time

# we use pathlib for platform-agnostic path handling 

from pathlib import Path

# we use YAML for runtime configuration files

import yaml

# We use logging for runtime logs

import logging

# demonext module

import demonext
from demonext import config, telescope, pdu, camera, focuser, obsfile, site

# Boolean state convenience translation dictionaries

OnOff = {True:'On',False:'Off'}
YesNo = {True:'Yes',False:'No'}

# platform-agnostic path definition relative to home

configDir = Path.home() / ".demonext/config"
configFile = "demonext.txt"

cfgFile = str(Path.home() / configDir / configFile)

# read by instantiating a demonext Config class

try:
    cfg = config.Config(cfgFile)
except Exception as exp:
    print(f"ERROR: (Config) - {exp}")

# start the runtime logger

logDir = demonext.homePath(cfg.config["directories"]["LogDir"])

logFile = str(Path(logDir) / f"eng{demonext.obsDate()}.txt")

logging.basicConfig(filename=logFile,
                    format="%(asctime)s %(name)s %(levelname)s: %(message)s",
                    filemode="a",
                    level=logging.INFO)

# ID for log entries

logger = logging.getLogger("darkTest")

logger.info("Started the DEMONEXT CCD dark calibration notebook")

## Startup the instrument systems we need

instantiate camera, telescope, focuser, and site classes

In [20]:
# Camera class 

try:
    cam = camera.Camera(cfgFile)
    print("Camera class started")
except Exception as exp:
    msg = f"Cannot initialize Camera instance: {exp}"
    print(f"ERROR: {msg}")
    logger.exception(msg)

# Telescope class

try:
    tel = telescope.Telescope(cfgFile)
    print("Telescope class started")
except Exception as exp:
    msg = f"Cannot initialize Telescope instance: {exp}"
    print(f"ERROR: {msg}")
    logger.exception(msg)

# Focuser class

try:
    foc = focuser.Focuser(cfgFile)
    print("Focuser class started")
except Exception as exp:
    msg = f"Cannot initialize Focuser instance: {exp}"
    print(f"ERROR: {msg}")
    logger.exception(msg)

# Site class

try:
    sro = site.Site(cfgFile)
    print("Site class started")
except Exception as exp:
    msg = f"Cannot initialize a Site instance: {exp}"
    print(f"ERROR: {msg}")
    logger.exception(msg)

# share the Telescope class object with the Camera class instance

cam.telescope = tel

# share the Focuser class object with the Camera class instance

cam.focuser = foc

# Share the Site class object with the Camera class object 

cam.site = sro

# Connect the services...

# connect to the telescope controller

logger.info("Connecting to telescope...")

print("Connecting to the telescope mount...")
try:
    tel.connect()
except Exception as exp:
    print(f"Cannot connect to telescope: {exp}")

if tel.connected:
    print(f"Done: Connected to {tel.telName} successfully")
else:
    print(f"Error: Failed to connect to the telescope controller")
    
# connect the camera

print("\nConnecting to the cameras...")
try:
    cam.connect()
except Exception as exp:
    print(f"Cannot connect to the cameras: {exp}")

if cam.connected:
    print(f"Done: Connected to MaxIM DL and the cameras")
else:
    print(f"Error: Failed to connect to MaxIM DL and the cameras")

# connect the focuser

print("\nConnecting to the focuser...")
try:
    foc.connect()
except Exception as exp:
    print(f"Cannot connect to the focuser: {exp}")

if foc.connected:
    print(f"Done: Connected to {foc.name}")
else:
    print(f"Error: Failed to connect to PWI3 and the Hedrick focusers")


Camera class started
Telescope class started
Focuser class started
Site class started


## Get camera and telescope information


In [26]:
# query mount information only, no moves yet

logger.info("querying CCD info")

ccdInfo = cam.getCCDInfo()
try:
    ccdInfo = cam.getCCDInfo()
except Exception as exp:
    print(f"Cannot get camera info: {exp}")

print("Science CCD Info:")
print(f"  Format: {ccdInfo['sizeX']}x{ccdInfo['sizeY']}")
print(f"  Pixel Size: {ccdInfo['pixSizeX']}x{ccdInfo['pixSizeY']}")
print(f"  Binning: {ccdInfo['binX']}x{ccdInfo['binY']}")
ccdXS = ccdInfo['startX']
ccdXE = ccdXS + ccdInfo['numX']
ccdYS = ccdInfo['startY']
ccdYE = ccdYS + ccdInfo['numY']
print(f"  Readout ROI: [{ccdXS}:{ccdXE},{ccdYS}:{ccdYE}]")

print("\nCCD Thermoelectric Cooler Status:")
print(f"     Cooler State: {ccdInfo['TECState']}, Cooler Power: {ccdInfo['TECPower']:.1f}%")
print(f"  CCD Temperature: {ccdInfo['CCDTemp']:.2f}C")
print(f"   Heat Sink Temp: {ccdInfo['TECTemp']:.2f}C")
print(f"    SetPoint Temp: {ccdInfo['setPoint']:.2f}C")
    
# telescope info

try:
    telInfo = tel.position()
except Exception as exp:
    print(f"Cannot get telescope position info: {exp}")

print(f"\nTelescope Position (decimal):")
print(f"  Alt/Az: {telInfo["Alt"]:.5f} d, {telInfo["Az"]:.5f} d")
print(f"  RA/Dec: {telInfo["RA"]:.5f} h, {telInfo["Dec"]:.5f} d")
print(f"  LST/HA: {telInfo["LST"]:.5f}h, {telInfo["HA"]:.5f} h")
print(f"    SecZ: {telInfo["SecZ"]:.2f}")

telFITS = tel.telFITS()
print(f"\nTelescope Position (FITS style):")
print(f"  Alt/Az: {telFITS["TELALT"]} {telFITS["TELAZ"]}")
print(f"  RA/Dec: {telFITS["TELRA"]} {telFITS["TELDEC"]}")
print(f"  LST/HA: {telFITS["LST"]} {telFITS["TELHA"]}")
print(f"    SecZ: {telFITS["SECZ"]:.2f}")

# quick tests of the decimal <-> sexigesimal conversion methods 

print("\nFormat Conversion:")
print(f"  sexigesimal -> decimal: {telFITS["TELRA"]} = {tel.sex2dec(telFITS["TELRA"]):.8f}")
print(f"  decimal -> sexigesimal: {telInfo["RA"]:.8f} = {tel.dec2sex(telInfo["RA"])}")

# Mount Status

print(f"\nTelescope Mount Status:")
print(f"   At Home? {YesNo[tel.isHome()]}")
print(f"    Parked? {YesNo[tel.isParked()]}")
print(f"  Tracking: {OnOff[tel.isTracking()]}")
print(f"   Slewing? {YesNo[tel.isSlewing()]}")

# Pointing limits

print(f"\nPointing Limits:")
print(f"   HA: {tel.minHA:.1f} to {tel.maxHA:.1f}h")
print(f"  Dec: {tel.minDec:.2f} to 0 d")
print(f"  Alt: {tel.minAlt:.1f} to 90 d")

# data directory info

print("\nData Directories and Filenames:")
print(f"  Base Data Directory: {cam.setDataDir()}")
print(f"  Observing Date: {cam.getObsDate()}")
for imgType in ['sci','bias','dark','flat','gcal']:
    print(f"  Next {imgType} file: {cam.getNextFile(imgType)}")

# direct cooling methods

print("\nCooling Info direct methods:")
print(f"  CCD Temp = {cam.ccdTemp():.2f} C")
print(f"  TEC Temp = {cam.tecTemp():.2f} C")
print(f"  SetPoint = {cam.setPoint():.1f} C")
print(f" TEC Power = {cam.tecPower()}%")
print(f"   Cooling: {cam.isCooling()} = {OnOff[cam.isCooling()]}")

Science CCD Info:
  Format: 2048x2064
  Pixel Size: 15.0x15.0
  Binning: 1x1
  Readout ROI: [0:2048,0:2064]

CCD Thermoelectric Cooler Status:
     Cooler State: Off, Cooler Power: 7.0%
  CCD Temperature: 21.75C
   Heat Sink Temp: 21.62C
    SetPoint Temp: -10.00C

Telescope Position (decimal):
  Alt/Az: 69.09208 d, 229.00566 d
  RA/Dec: 2.38293 h, 22.10572 d
  LST/HA: 3.50972h, 1.12679 h
    SecZ: 1.07

Telescope Position (FITS style):
  Alt/Az: +69:05:31.487 229:00:20.390
  RA/Dec: 02:22:58.627 +22:06:20.583
  LST/HA: 03:30:34.992 +01:07:36.365
    SecZ: 1.07

Format Conversion:
  sexigesimal -> decimal: 02:22:58.627 = 2.38295194
  decimal -> sexigesimal: 2.38293426 = 02:22:58.56

Telescope Mount Status:
   At Home? No
    Parked? No
  Tracking: Off
   Slewing? No

Pointing Limits:
   HA: -11.0 to 11.0h
  Dec: -47.93 to 0 d
  Alt: 5.0 to 90 d

Data Directories and Filenames:
  Base Data Directory: D:\Data
  Observing Date: 20260408
  Next sci file: D:\Data\20260408\sci20260408.0001.f

## Camera cooldown

Enable camera cooling and wait until it reaches the base set point of 0C

In [ ]:
# cool the camera to the setpoint

t0 = time.time()
print(f"Cooling down the CCD camera...")

Temp0 = 0.0

try:
    cam.cooldown(Temp0)
except Exception as exp:
    print(f"oops: {exp}")

print(f"Done: CCD camera temperature is {cam.ccdTemp():.1f}C")
dt = (time.time() - t0)/60.0
print(f"      cooldown time: {dt:.1f} minutes")

print(f"Base CCD temperature set to 0.0C, ready to begin test")


### Take a Dark sequence

Set CCD temperature, followed by 5 bias and 5 darks of 240s duration, for a range of temperatures
from -5 to -25C in steps of 5C.

In [38]:
numBias = 5
numDarks = 5
expTime = 240.0

for temp in [-5.0,-10.0,-15.0,-20.0,-25.0]:
    print(f"Cooling down the CCD camera to {temp:.1} C...")

    try:
        cam.cooldown(temp)
    except Exception as exp:
        print(f"oops: {exp}")

    print(f"Done: CCD camera temperature is {cam.ccdTemp():.1f}C")

    print(f"Taking {numBias} bias images...")
    try:
        cam.bias(nimgs=numBias)
    except Exception as exp:
        print(f"oops: {exp}")
    print(f"Done: last bias image {cam.nextFile}")

    print(f"Taking {numDarks} {expTime:.1f} sec dark images...")
    try:
        cam.dark(expTime,nimgs=numDarks)
    except Exception as exp:
        print(f"oops: {exp}")
    print(f"Done: last dark image {cam.nextFile}")

print("Done: dark temp sequence completed")


Taking 4 bias images...
Done: last bias image D:\Data\20260408\bias20260408.0004.fits
      execution time: 18.945 seconds


## Camera warm-up

Controlled warm-up of the the camera to ambient.

In [ ]:
# warm the camera to ambient

t0 = time.time()
print(f"Warming the CCD camera to ambient...")

try:
    cam.warmup()
except Exception as exp:
    print(f"oops: {exp}")

print(f"Done: CCD camera temperature is {cam.ccdTemp():.1f}C")
dt = (time.time() - t0)/60.0
print(f"      warmup time: {dt:.1f} minutes")
